# B3 · Rare classes: weights, `balanced_subsample` and a threshold chosen with the OOB

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

On a problem with 3% of positives compare four forests (base, `class_weight="balanced"`, `"balanced_subsample"`, minimum leaf 10) with average precision and F1; choose the **threshold** from the `oob_decision_function_` (without cross-validation and without touching the test set) and check that it improves the F1 over 0.5;
measure the effect on the mean probabilities.

## Dataset

`make_classification(12000, 12 features, 5 informative, about 3% positives)`, stratified 50/50 split: `X_tr, y_tr, X_te, y_te` (204 positives in each).

## Educational purpose

Exploit the peculiarity of the forest: the **out-of-bag probabilities** are out-of-sample predictions for the whole training set, so a threshold can be chosen on them without cross-validation and without contaminating the test set. And see what class weights really do in a forest.

## Task

1. For each of the four configurations (`RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1, ...)`: `base`, `balanced`, `balanced_subsample`, `leaf10` = `min_samples_leaf=10`) save on the test set: `ap` (average precision), `auc`, `f1_05` (F1 with threshold 0.5) and `mean_p` (mean predicted probability). Build `table` (index: configurations).
2. Write `f1_threshold(y, p)` (grid `np.linspace(0.02, 0.9, 89)`, threshold with maximum F1). For each configuration compute `oob_threshold` from the OOB probabilities (`oob_decision_function_[:, 1]`) and `oob_f1` (F1 on the **test set** with that threshold). Add the two columns to `table`.
3. Save `best_f1` (configuration with the highest `oob_f1`) and `threshold_gain` = `oob_f1 − f1_05` for `base`.
4. Save `threshold_far_from_05` (True if the `oob_threshold` of `base` is smaller than 0.4).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(12000, n_features=12, n_informative=5, n_redundant=0, weights=[0.97], class_sep=0.9, flip_y=0.01, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)
configs = {"base": {}, "balanced": {"class_weight": "balanced"}, "balanced_subsample": {"class_weight": "balanced_subsample"}, "leaf10": {"min_samples_leaf": 10}}
print("positives: training", int(y_tr.sum()), "| test", int(y_te.sum()))

## Your work

In [ ]:
def f1_threshold(y, p):
    ...

In [ ]:
rows = {}
for name, kw in configs.items():
    f = RandomForestClassifier(300, oob_score=True, random_state=0, n_jobs=-1, **kw).fit(X_tr, y_tr)
    ...
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
best_f1 = None
threshold_gain = None
threshold_far_from_05 = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`p_oob = f.oob_decision_function_[:, 1]`; `threshold = f1_threshold(y_tr, p_oob)`; `f1_score(y_te, p_test >= threshold)`. With `oob_score=True` the attribute `oob_decision_function_` exists after `fit`.'):
    check(list(table.columns) == ["ap", "auc", "f1_05", "mean_p", "oob_threshold", "oob_f1"] and list(table.index) == list(configs), "table with six columns and four configurations", "ap, auc, f1_05, mean_p, oob_threshold, oob_f1")
    check(table["auc"].min() > 0.83 and table["ap"].min() > 0.5 and table["ap"].max() - table["ap"].min() < 0.06, "AUC over 0.83 and average precision 0.5-0.65 for all: the configurations rank almost the same way", "the forest already handles the imbalance well")
    check((table["oob_f1"] > table["f1_05"]).all(), "with the OOB threshold the F1 on the test set improves over threshold 0.5 for all the configurations", "oob_decision_function_[:, 1]")
    check(table.loc["base", "oob_threshold"] < 0.4 and threshold_far_from_05 is True and threshold_gain > 0.04, "the OOB threshold for the base forest is below 0.4 and gains over 0.04 of F1", "oob_f1 - f1_05 for the base configuration")
    check(best_f1 in configs and table.loc[best_f1, "oob_f1"] == table["oob_f1"].max(), "configuration with the highest F1", "idxmax of oob_f1")
    check(table.loc["base", "f1_05"] > table.loc["balanced", "f1_05"], "at threshold 0.5 the 'balanced' weights worsen the F1 (the base model has a much higher precision)", "compare f1_05")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_imbalanced_forest_threshold_from_oob.ipynb`. Open it after you pass the check, or after you have really tried.